<a href="https://colab.research.google.com/github/scalesynthai/phdai733-group8/blob/main/part1/notebooks/Group8_Part1_Final_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting Heart Disease Risk: A Clinical Decision-Support Prototype

### PhDAI 733-A02 — Group Project Part 1
**Group 8:** Subba Taniparti · Harini Mamidala · Christian Gaston · Naveen Vishal Bellary

**Instructor:** Dr. Karriem Perry | **Due:** September 27, 2026

---

## Problem Framing

Cardiovascular disease is the leading cause of death worldwide, and early identification of at-risk
patients materially changes outcomes. The clinical difficulty is not treatment but triage: a
cardiologist referral and an angiogram are expensive and invasive, so referring every patient
presenting with chest pain is not viable, while referring too few means missed disease.

This project builds a decision-support prototype that estimates heart-disease risk from thirteen
routinely collected clinical measurements — the kind available from a standard workup without
invasive testing. The intended use is **triage support, not diagnosis**: flagging patients who
warrant further cardiac investigation.

**Research questions**

- **RQ1.** Can routinely collected clinical measurements predict heart disease substantially better
  than a majority-class baseline?
- **RQ2.** Does a flexible ensemble (random forest) outperform a regularized linear model
  (logistic regression) on a dataset of this size, and is any difference stable?
- **RQ3.** Does model performance differ systematically between patient sex subgroups, and if so,
  what explains it?

**Hypotheses**

- **H1.** Both models will substantially exceed the majority-class baseline, since the predictors
  are established clinical correlates of cardiac disease.
- **H2.** Logistic regression will be competitive with or better than random forest, because 302
  patients across 22 encoded features gives a high-variance ensemble limited signal to exploit.
- **H3.** Performance will differ between sex subgroups, driven substantially by differing disease
  prevalence rather than by the model attending to sex directly.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     StratifiedKFold, cross_val_score)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix,
                             classification_report)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid'); sns.set_palette('muted')
pd.set_option('display.max_columns', None)
RANDOM_STATE = 42


## 1. Dataset Exploration

**Source:** Heart Disease dataset, Cleveland subset, UCI Machine Learning Repository
(Janosi et al., 1989), accessed via an open GitHub mirror on September 23, 2026.

The target indicates presence of heart disease. Thirteen predictors cover demographics (age, sex),
symptom presentation (chest pain type, exercise-induced angina), resting measurements (blood
pressure, cholesterol, fasting blood sugar, resting ECG), and stress-test results (maximum heart
rate, ST depression, ST slope, fluoroscopy vessel count, thalassemia scan).


In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/sharmaroshan/Heart-UCI-Dataset/master/heart.csv'
df_raw = pd.read_csv(DATA_URL)

print(f"Raw shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print(f"Missing values: {df_raw.isna().sum().sum()}")
print(f"Duplicate rows: {df_raw.duplicated().sum()}")
df_raw.head()


In [ ]:
df_raw.describe().T.round(2)


### 1.1 A structural issue the summary statistics conceal

`.describe()` reports every column as numeric, which is misleading. Five predictors are
**categorical codes stored as integers**. Chest pain type (`cp`) takes values 0–3 representing
four distinct clinical presentations; the value 3 is not "three times" the value 1.

This matters practically: `pd.get_dummies(df)` operates on object-dtype columns, so calling it on
this dataframe encodes nothing and returns the data unchanged, with no warning.


In [ ]:
print(f"Columns pandas treats as categorical (object dtype): {(df_raw.dtypes == object).sum()}")
print("=> pd.get_dummies(df_raw) would encode 0 columns and appear to succeed.\n")

profile = pd.DataFrame({'unique_values': df_raw.nunique(),
                        'dtype': df_raw.dtypes.astype(str),
                        'min': df_raw.min(), 'max': df_raw.max()})
profile['treat_as_categorical'] = (profile['unique_values'] <= 5) & (profile.index != 'target')
profile


### 1.2 Validating the label encoding

Before modelling we verified what the `target` column actually encodes. This dataset circulates in
multiple mirrored versions and the target convention is not consistent between them, so the
encoding was checked against clinical expectation rather than assumed.

Four predictors have unambiguous directional relationships with coronary disease in the clinical
literature: patients with disease achieve a **lower** maximum heart rate (diseased arteries limit
exertion), show **greater** ST depression (an ischaemia marker), have **more** major vessels
visible on fluoroscopy, and more often present with exercise-induced angina.


In [ ]:
check = df_raw.drop_duplicates().groupby('target')[['thalach', 'oldpeak', 'ca', 'exang']].mean().round(2)
print(check.to_string())
print()
print("Expected direction for the DISEASED group:")
print("  thalach lower | oldpeak higher | ca higher | exang higher")
print()
for feat, expect in [('thalach','lower'), ('oldpeak','higher'), ('ca','higher'), ('exang','higher')]:
    v0, v1 = check.loc[0, feat], check.loc[1, feat]
    diseased = 0 if ((v0 < v1) if expect == 'lower' else (v0 > v1)) else 1
    print(f"  {feat:8s}: target=0 -> {v0:6.2f}, target=1 -> {v1:6.2f}  => diseased group is target={diseased}")


**Finding.** All four markers agree: **`target = 0` denotes disease and `target = 1` denotes its
absence.** This is the opposite of the convention most commonly assumed for this dataset, and
accepting the default reading would have inverted every clinical conclusion in this report — the
confusion matrix axes, the meaning of recall, and the entire fairness analysis.

We therefore define an explicit `disease` variable and use it as the target throughout. This check
cost a few lines of code and is the single most consequential preprocessing step we took.


## 2. Preprocessing

Four decisions, each justified rather than applied by default.


In [ ]:
# 2.1 Remove the exact duplicate. A repeated patient record would appear in both
# train and test after splitting, inflating apparent performance.
df = df_raw.drop_duplicates().reset_index(drop=True)
print(f"Duplicates removed: {df_raw.shape[0] - df.shape[0]}  ->  {df.shape[0]} patients")

# Apply the corrected label established in Section 1.2
df['disease'] = 1 - df['target']
print(f"Disease prevalence: {df['disease'].mean():.3f} "
      f"({df['disease'].sum()} of {len(df)} patients)")

# 2.2 No imputation is required: the dataset is complete. The hints document suggests
# forward-fill, which would be inappropriate here anyway -- it assumes row order carries
# meaning, and these records are not ordered in any clinically meaningful way.
print(f"Missing values requiring imputation: {df.isna().sum().sum()}")


In [ ]:
# 2.3 Encode the categorical codes properly
CATEGORICAL = ['cp', 'restecg', 'slope', 'thal', 'ca']
BINARY      = ['sex', 'fbs', 'exang']
CONTINUOUS  = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

X = pd.get_dummies(
    df.drop(columns=['target', 'disease']).astype({c: 'category' for c in CATEGORICAL}),
    columns=CATEGORICAL, drop_first=True)
y = df['disease']

print(f"Encoded: {df.shape[1] - 2} predictors -> {X.shape[1]} features")
print(f"Created: {[c for c in X.columns if c not in df.columns]}")


**On scaling (a deliberate departure from the assignment hints).** The hints document applies
`StandardScaler` to the full dataset before `train_test_split`. Doing so fits the scaler's means
and standard deviations using test-set rows, leaking information from data the model is supposed
not to have seen and producing optimistic performance estimates.

We instead place the scaler inside a `Pipeline`, so it is fitted only on training folds during
cross-validation and only on the training set for the final fit. Scaling is applied to the five
continuous predictors; the binary and one-hot columns are passed through unchanged, since
standardizing an indicator variable distorts its interpretation without benefit.

This decision was taken by the team and is recorded in `DECISIONS.md` in our repository.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

print(f"Training: {X_train.shape[0]} patients | Test: {X_test.shape[0]} patients")
print(f"Disease prevalence -- train {y_train.mean():.3f}, test {y_test.mean():.3f} (stratified)")

preprocessor = ColumnTransformer([('scale', StandardScaler(), CONTINUOUS)],
                                 remainder='passthrough')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


### 2.1 Exploratory visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

sns.countplot(data=df, x='disease', ax=axes[0])
axes[0].set_title('Class balance'); axes[0].set_xlabel('Heart disease (0 = absent, 1 = present)')
axes[0].set_ylabel('Patients')

sns.boxplot(data=df, x='disease', y='thalach', ax=axes[1])
axes[1].set_title('Maximum heart rate by outcome')
axes[1].set_xlabel('Heart disease'); axes[1].set_ylabel('Max heart rate achieved (bpm)')

sns.boxplot(data=df, x='disease', y='oldpeak', ax=axes[2])
axes[2].set_title('ST depression by outcome')
axes[2].set_xlabel('Heart disease'); axes[2].set_ylabel('ST depression (oldpeak)')

for ax, lab in zip(axes, ['A', 'B', 'C']):
    ax.text(-0.12, 1.09, lab, transform=ax.transAxes, fontsize=13, fontweight='bold', va='top')

plt.tight_layout(); plt.savefig('fig1_eda.png', dpi=130, bbox_inches='tight'); plt.show()

print(f"Class balance: {y.value_counts().to_dict()} ({y.mean():.1%} have disease)")
for c in ['thalach', 'oldpeak']:
    g = df.groupby('disease')[c].mean()
    print(f"{c}: no disease {g[0]:.2f}, disease {g[1]:.2f}")


**Interpretation.** Panel A shows the classes are reasonably balanced (45.7% of patients have
disease), so accuracy is usable as a headline metric and resampling is unnecessary. It remains
insufficient on its own, however: in a triage application a missed case and a false alarm carry very
different costs, so recall and the confusion matrix matter more than a single accuracy figure.

Panels B and C use identical plot types and axis conventions, so the two predictors are directly
comparable. Both separate the classes visibly and in the clinically expected direction: patients
with disease achieve a *lower* maximum heart rate (139.10 vs 158.38 bpm) and show *greater* ST
depression (1.59 vs 0.59). These are the same relationships used in Section 1.2 to establish the
label encoding, and seeing them hold here confirms that correction was applied correctly.

The separation is clear but the distributions overlap substantially, which signals in advance that
no single predictor will suffice and a multivariate model is warranted.


In [ ]:
corr = df.drop(columns='target').corr()

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))

sns.heatmap(corr, annot=False, cmap='RdBu_r', center=0, square=True,
            linewidths=0.4, ax=axes[0], cbar_kws={'shrink': 0.8})
axes[0].set_title('Correlation matrix (all variables)')

target_corr = corr['disease'].drop('disease').sort_values()
colors = ['#C0392B' if v < 0 else '#2E86AB' for v in target_corr]
axes[1].barh(target_corr.index, target_corr.values, color=colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Correlation with heart disease')
axes[1].set_xlabel('Pearson r')

for ax, lab in zip(axes, ['A', 'B']):
    ax.text(-0.12, 1.08, lab, transform=ax.transAxes, fontsize=13, fontweight='bold', va='top')

plt.tight_layout(); plt.savefig('fig2_corr.png', dpi=130, bbox_inches='tight'); plt.show()
print(target_corr.round(3).to_string())


**Interpretation.** Panel B ranks each variable's linear association with disease, and the
ordering is clinically coherent: exercise-induced angina, ST depression, and fluoroscopy vessel
count are the strongest positive correlates of disease, while maximum heart rate and chest pain
type correlate negatively. Every one of these directions matches cardiology, which is a second
confirmation that the label correction in Section 1.2 was right.

Two observations matter for modelling. First, no single predictor exceeds |*r*| = .45, so
performance must come from combining signals rather than from any dominant variable. Second, Panel A
shows no block of near-collinear predictors of the kind that destabilises regression coefficients.
This supports retaining all thirteen predictors rather than pruning.

## 3. Model Development

Two models are compared against a majority-class baseline. The baseline is not a formality: with
45.7% of patients having disease, a classifier predicting "no disease" for everyone scores 54.3%
accuracy while identifying not a single case. Any reported accuracy must be read against that floor.

**Model selection rationale.** Logistic regression is included because it produces calibrated
probabilities and interpretable coefficients — a clinician can be shown which factors drove a
prediction, which matters for adoption in a clinical setting. Random forest is included because it
captures non-linear interactions without their being specified in advance, a plausible advantage
given that cardiac risk factors are known to interact.

`roc_auc` is the tuning criterion rather than accuracy, because it evaluates ranking quality across
all decision thresholds. For a triage tool the operating threshold should ultimately be set by
clinical cost, not left at 0.5, so a model that ranks patients well is more valuable than one
optimised at a single arbitrary cut-point.


In [ ]:
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)

logreg = GridSearchCV(
    Pipeline([('pre', preprocessor),
              ('clf', LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))]),
    param_grid={'clf__C': [0.01, 0.1, 1, 10, 100]},
    cv=cv, scoring='roc_auc', n_jobs=-1).fit(X_train, y_train)

rf = GridSearchCV(
    Pipeline([('pre', preprocessor),
              ('clf', RandomForestClassifier(random_state=RANDOM_STATE))]),
    param_grid={'clf__n_estimators': [100, 300],
                'clf__max_depth': [3, 5, None],
                'clf__min_samples_leaf': [1, 3, 5]},
    cv=cv, scoring='roc_auc', n_jobs=-1).fit(X_train, y_train)

print(f"Logistic regression -- best: {logreg.best_params_}")
print(f"  cross-validated ROC-AUC: {logreg.best_score_:.4f}")
print(f"\nRandom forest -- best: {rf.best_params_}")
print(f"  cross-validated ROC-AUC: {rf.best_score_:.4f}")


In [ ]:
def evaluate(model, name):
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    return {'Model': name,
            'Accuracy': accuracy_score(y_test, pred),
            'Precision': precision_score(y_test, pred, zero_division=0),
            'Recall': recall_score(y_test, pred, zero_division=0),
            'F1': f1_score(y_test, pred, zero_division=0),
            'ROC-AUC': roc_auc_score(y_test, proba) if proba is not None else np.nan}

results = pd.DataFrame([evaluate(baseline, 'Baseline (majority class)'),
                        evaluate(logreg, 'Logistic regression (tuned)'),
                        evaluate(rf, 'Random forest (tuned)')])
results.round(4)


In [ ]:
print(classification_report(y_test, logreg.predict(X_test),
                            target_names=['No disease', 'Disease'], digits=3))


### 3.1 Is the difference stable, or an artifact of one split?

The test set contains 76 patients, so a single split supports only weak conclusions — a handful of
patients changing side moves accuracy by several points. Both models are therefore refitted across
30 independent stratified splits.


In [ ]:
def make_logreg():
    return Pipeline([('pre', preprocessor),
                     ('clf', LogisticRegression(C=logreg.best_params_['clf__C'],
                                                max_iter=5000, random_state=RANDOM_STATE))])

def make_rf():
    p = {k.replace('clf__', ''): v for k, v in rf.best_params_.items()}
    return Pipeline([('pre', preprocessor),
                     ('clf', RandomForestClassifier(random_state=RANDOM_STATE, **p))])

N_SPLITS = 30
stability = {'Logistic regression': [], 'Random forest': []}

for seed in range(N_SPLITS):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
    for name, maker in [('Logistic regression', make_logreg), ('Random forest', make_rf)]:
        stability[name].append(accuracy_score(yb, maker().fit(Xa, ya).predict(Xb)))

lr_wins = sum(1 for i in range(N_SPLITS)
              if stability['Logistic regression'][i] > stability['Random forest'][i])

for name, vals in stability.items():
    v = np.array(vals)
    print(f"{name}: mean accuracy {v.mean():.4f} (SD {v.std():.4f}), range {v.min():.3f}-{v.max():.3f}")
print(f"\nLogistic regression outperformed random forest in {lr_wins} of {N_SPLITS} splits.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

for ax, (name, model) in zip(axes, [('Logistic regression (tuned)', logreg),
                                    ('Random forest (tuned)', rf)]):
    cm = confusion_matrix(y_test, model.predict(X_test))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['No disease', 'Disease'],
                yticklabels=['No disease', 'Disease'],
                vmin=0, vmax=cm.max(), annot_kws={'size': 13})
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f"{name}\nAccuracy = {accuracy_score(y_test, model.predict(X_test)):.3f}")

for ax, lab in zip(axes, ['A', 'B']):
    ax.text(-0.14, 1.10, lab, transform=ax.transAxes, fontsize=13, fontweight='bold', va='top')

plt.tight_layout(); plt.savefig('fig3_confusion.png', dpi=130, bbox_inches='tight'); plt.show()

for name, model in [('Logistic regression', logreg), ('Random forest', rf)]:
    tn, fp, fn, tp = confusion_matrix(y_test, model.predict(X_test)).ravel()
    print(f"{name}: missed cases (FN) = {fn}, false alarms (FP) = {fp}")


**Interpretation.** Both panels use an identical layout, class ordering, and colour scale, so
the only quantity varying between them is the model.

The bottom-left cell is the clinically important one: patients who have disease but were classified
as healthy. Logistic regression places 7 patients there against the random forest's 10. In a triage
application these errors are not interchangeable with the top-right cell — a false alarm (6 and 7
respectively) costs an unnecessary consultation, whereas a missed case may mean untreated coronary
disease.

The stability analysis supports preferring logistic regression: it achieved higher accuracy in 26
of 30 independent splits (mean 84.3% vs 80.4%), so its advantage is a property of the model on this
data rather than an artifact of one partition. That the simpler model wins is consistent with H2 —
226 training patients across 22 features give a high-variance ensemble little opportunity to exploit
its extra flexibility. Cross-validated AUC was close (.911 vs .904), but the gap widened on held-out
data (.926 vs .867), the signature of the ensemble fitting training-set noise.


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.2))
for name, model in [('Logistic regression', logreg), ('Random forest', rf)]:
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {roc_auc_score(y_test, proba):.3f})")
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Chance')
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
ax.set_title('ROC curves on held-out test set'); ax.legend(loc='lower right', fontsize=9)
plt.tight_layout(); plt.savefig('fig4_roc.png', dpi=130, bbox_inches='tight'); plt.show()


## 4. Ethics & Impact

The assignment brief asks whether the model performs differently across demographic groups.
Answering that requires disaggregating performance rather than reporting it in aggregate.

The dataset records patient sex. It is coded 0/1 without documentation of which value denotes which
sex, so the groups are reported below as `sex = 0` and `sex = 1` and no claim is made about which
biological population each represents. The audit can establish *that* performance differs; it
cannot attribute that difference to a named group.


In [ ]:
audit = []
for name, model in [('Logistic regression', logreg), ('Random forest', rf)]:
    pred = model.predict(X_test); proba = model.predict_proba(X_test)[:, 1]
    for grp in [0, 1]:
        mask = (X_test['sex'] == grp).values
        tn, fp, fn, tp = confusion_matrix(y_test[mask], pred[mask], labels=[0, 1]).ravel()
        audit.append({'Model': name, 'Group': f'sex = {grp}', 'n': int(mask.sum()),
                      'Prevalence': y_test[mask].mean(),
                      'Accuracy': accuracy_score(y_test[mask], pred[mask]),
                      'Recall': recall_score(y_test[mask], pred[mask], zero_division=0),
                      'Missed cases (FN)': int(fn), 'False alarms (FP)': int(fp)})

audit_df = pd.DataFrame(audit)
audit_df.round(3)


In [ ]:
# Is the recall gap stable, or noise from ~26 patients per subgroup?
gaps = []
for seed in range(N_SPLITS):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
    model = make_logreg().fit(Xa, ya); pred = model.predict(Xb)
    r = {}
    for grp in [0, 1]:
        mask = (Xb['sex'] == grp).values
        if mask.sum() >= 5:
            r[grp] = recall_score(yb[mask], pred[mask], zero_division=0)
    if len(r) == 2:
        gaps.append(r[0] - r[1])

gaps = np.array(gaps)
print(f"Recall gap (sex=0 minus sex=1) across {len(gaps)} splits:")
print(f"  mean {gaps.mean():+.3f}, SD {gaps.std():.3f}")
print(f"  sex=0 recall LOWER in {(gaps < 0).sum()} of {len(gaps)} splits")
print(f"  sex=0 recall higher in {(gaps > 0).sum()} of {len(gaps)} splits")
print(f"\nDisease prevalence -- sex=0: {df[df.sex==0]['disease'].mean():.3f}, "
      f"sex=1: {df[df.sex==1]['disease'].mean():.3f}")
print(f"Subgroup sizes -- sex=0: {(df.sex==0).sum()}, sex=1: {(df.sex==1).sum()}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))

pivot = audit_df.pivot(index='Group', columns='Model', values='Recall')
pivot.plot(kind='bar', ax=axes[0], width=0.75)
axes[0].set_ylabel('Recall (sensitivity)'); axes[0].set_xlabel('')
axes[0].set_title('Recall by subgroup and model'); axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(fontsize=8); axes[0].set_ylim(0, 1.05)

axes[1].hist(gaps, bins=12, alpha=0.85, edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.4, label='No gap')
axes[1].axvline(gaps.mean(), color='#C0392B', linewidth=2, label=f'Mean = {gaps.mean():+.2f}')
axes[1].set_xlabel('Recall gap (sex = 0 minus sex = 1)')
axes[1].set_ylabel('Number of splits')
axes[1].set_title(f'Gap across {len(gaps)} splits'); axes[1].legend(fontsize=8)

for ax, lab in zip(axes, ['A', 'B']):
    ax.text(-0.10, 1.10, lab, transform=ax.transAxes, fontsize=13, fontweight='bold', va='top')

plt.tight_layout(); plt.savefig('fig5_fairness.png', dpi=130, bbox_inches='tight'); plt.show()


**Interpretation (RQ3/H3).** Panel A shows a pattern that a single aggregate metric would
have hidden entirely, and it is the central ethical finding of this project.

For the `sex = 0` subgroup the model achieves **higher accuracy** (.917 vs .788) but **lower
recall** (.714 vs .821). Those two facts point in opposite directions, and recall is the one that
matters clinically: of the 7 patients in that subgroup who actually had disease, 2 were classified
as healthy. Reporting accuracy alone would have made this subgroup look like the model's *best*
case.

**The mechanism is prevalence, not the sex feature.** Disease prevalence is 25.0% among `sex = 0`
patients versus 55.3% among `sex = 1`. In a low-prevalence group a classifier earns high accuracy
cheaply by leaning toward the negative class, and that same lean suppresses recall. Compounding it,
`sex = 0` contributes only 96 of 302 patients — roughly 24 disease cases in the entire dataset —
so the model has little data from which to learn that subgroup's presentation.

**Honesty about strength of evidence.** Panel B shows the recall gap across 30 splits: `sex = 0`
recall is lower in 19 of 30 splits (higher in 10, tied in 1), with a mean gap of -0.090 and a standard deviation of 0.184. The
direction is consistent more often than not, but the variance is large because each split leaves
only about 7 disease cases in that subgroup. We therefore report this as a **credible signal
warranting investigation, not an established bias**. Claiming statistical certainty from roughly 7
cases per split would misrepresent the evidence.

**Why "fairness through unawareness" would not help.** Removing `sex` from the feature set would
not correct the disparity, because the prevalence imbalance would persist and the remaining clinical
predictors correlate with sex. It would only remove our ability to detect the problem.

**Impact if deployed.** Lower recall means more missed cases. A tool with this profile would
under-refer `sex = 0` patients relative to their actual need, while its high accuracy in that
subgroup made it appear to be performing well. This is the practical argument for putting
disaggregated evaluation inside the model selection procedure rather than treating it as an optional
final check.

**Further considerations.** This is a 1989 single-institution Cleveland cohort; demographics and
clinical practice have changed substantially, so transferability to a contemporary population should
not be assumed. The operating threshold is a clinical judgment about the relative cost of a missed
case versus an unnecessary referral — not a statistical one, and not ours to make.

## 5. Findings

- **RQ1/H1 — supported.** Both models substantially exceed the 53.9% majority-class baseline. Tuned
  logistic regression reaches 82.9% accuracy and .926 ROC-AUC on held-out data.
- **RQ2/H2 — supported.** Logistic regression outperformed random forest in 26 of 30 splits
  (mean 84.3% vs 80.4%) and missed fewer cases on the primary split (7 vs 10), consistent with the
  ensemble's flexibility offering little benefit at this sample size.
- **RQ3/H3 — partially supported.** Performance does differ across sex subgroups, and the mechanism
  is prevalence as hypothesised (25.0% vs 55.3%) rather than direct use of the sex feature. The
  direction is consistent in 19 of 30 splits, but subgroup samples of roughly 7 disease cases make
  the magnitude uncertain, so we report a credible signal rather than a confirmed disparity.
- **Most consequential single step:** validating the label encoding (Section 1.2). The common
  assumption about this dataset's target is inverted; accepting it would have reversed every
  clinical conclusion here.

## 6. Limitations

With 302 patients and a 76-patient test set, all estimates carry wide sampling uncertainty, which is
why conclusions rest on 30-split replication rather than single-split significance testing. The
subgroup analysis is the weakest link: `sex = 0` contributes roughly 24 disease cases in total and
about 7 per test split, enough to detect a large disparity but not to quantify a moderate one, and
this is reflected in the 0.184 standard deviation of the recall gap. The data is a single-
institution cohort collected in 1989, limiting generalisability to contemporary populations and
practice patterns. Hyperparameter grids are coarse, though the consistency of the model ranking
across 30 splits suggests a finer search would not reverse it. All relationships reported are
predictive rather than causal: nothing here licenses claims that intervening on a predictor would
change patient outcomes.

## References

Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1989). *Heart disease* [Data set]. UCI
Machine Learning Repository. https://doi.org/10.24432/C52P4X

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, E.
(2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*,
2825–2830.
